# AICIDENTIA - 04 RQ2: network of incidents, organisations, harmed groups and risk domains

**RQ2:** Which organisations, AI systems and affected groups sit at the centre of the incident network, and what communities of related risk emerge? Compare Louvain vs Leiden vs label propagation; analyse centrality.

> CRISP-DM phases: **Modeling** (graph construction, community detection) and **Evaluation** (modularity, stability, robustness)

Logic lives in `aicidentia.models.network` and `aicidentia.models.network_outputs`; the write-up is `docs/rq2_network.md`. Organisations are **alleged** deployers and developers as AIID records them: centrality is frequency of involvement in reported incidents, never blame. Person-like entities are excluded from the network.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
pd.set_option("display.width", 220, "display.max_colwidth", 70)
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Build the networks

Two views: a typed heterogeneous graph (incidents, organisations, AI systems, harmed groups, risk domains, sectors) and a weighted **entity co-occurrence network** (weight = incidents two entities share). Centrality and communities use the co-occurrence network, restricted to entities in at least two incidents.

In [ ]:
from aicidentia.models import network_outputs

res = network_outputs.run_network()
network_outputs.write_all(res)
f = res.facts
print(
    "heterogeneous graph:",
    f["hetero_nodes"],
    "nodes,",
    f["hetero_edges"],
    "edges;",
    f["hetero_nodes_by_type"],
)
print(
    "entity network:",
    f["entity_network_nodes"],
    "nodes,",
    f["entity_network_edges"],
    "edges,",
    f["entity_network_components"],
    "components",
)
print("excluded person-like mentions:", f["excluded_person_like_mentions"])

## 2. Centrality

Degree, weighted degree, exact betweenness (inverse-weight lengths), PageRank and eigenvector centrality. The measures answer different questions (partner diversity, intensity, bridging, influence), so agreement between them is moderate.

In [ ]:
display(res.tau_all.round(2))
for node_type in ["organisation", "ai_system", "harmed_group"]:
    print(node_type)
    display(res.tops["pagerank"][node_type].round(4))
display(Image(filename=FIG / "fig_rq2_2_top_central.png"))
display(res.projection.head(10))

## 3. Communities: three algorithms, ten seeds

Chosen by a rule fixed in advance (best mean modularity among methods with mean ARI >= 0.5). Label propagation collapses on this hub-heavy network. Communities are formed from entity ties only; risk domains and sectors are used afterwards to *profile* them.

In [ ]:
display(res.method_summary.round(3))
print(
    "chosen:",
    res.chosen_method,
    "|",
    res.chosen_reason,
    "| near tie:",
    res.near_tie,
    "| medoid seed:",
    res.medoid_seed,
)
display(Image(filename=FIG / "fig_rq2_4_community_methods.png"))
display(
    res.profile[
        [
            "community",
            "entities",
            "incidents",
            "organisations",
            "ai_systems",
            "harmed_groups",
            "domains",
        ]
    ]
)
print("association with MIT domain (NMI):", round(res.association["nmi_community_vs_domain"], 3))
display(Image(filename=FIG / "fig_rq2_1_entity_network.png"))

**Robustness.** Removing the generic hub entities (present in more than 5% of incidents) changes the partition a lot, so community boundaries are hub-dependent; treat them as one descriptive reading.

In [ ]:
print({k: v for k, v in res.robustness.items() if k != "hubs_removed"})
print("hubs removed:", res.robustness["hubs_removed"])

## 4. Temporal evolution

Cumulative snapshots at the end of each year (partial year is the last row).

In [ ]:
display(res.snapshots[["year", "nodes", "edges", "organisations", "new_organisations"]])
display(Image(filename=FIG / "fig_rq2_5_network_growth.png"))
display(res.overlap.round(2))
display(Image(filename=FIG / "fig_rq2_6_top_organisations_over_time.png"))

## 5. Dashboard export and registry

In [ ]:
print(f["export"])
for key in [
    "rq2.chosen_method",
    "rq2.n_communities",
    "rq2.chosen_modularity",
    "rq2.nmi_community_vs_domain",
]:
    print(key, "=", registry.get(key))